# 001 — Imbalanced Classification

Comparing strategies for handling class imbalance in binary classification.
We evaluate six resampling and weighting strategies (no handling, class weights,
SMOTE, ADASYN, random undersampling, and SMOTEENN) using a configurable base
learner with Optuna tuning. Primary metric is PR-AUC, with full threshold
analysis and SHAP explanations for the best strategy.

**Lifecycle stage:** seedling (model-garden)

All code is self-contained in this notebook — no external library imports
from a shared `src/` package.

In [ ]:
# ---------------------------------------------------------------------------
# Papermill parameters  (this cell is tagged "parameters")
# ---------------------------------------------------------------------------

# Data loading
feature_paths: list[str] = []          # local or gs:// URIs; empty -> synthetic
target_col: str = 'target'
positive_label: str | int | None = None  # Recode this value to 1; None -> use as-is

# Synthetic data
imbalance_ratio: float = 0.05          # positive class ratio for synthetic data

# Splitting
test_size: float = 0.2
random_state: int = 42

# Model
base_model: str = 'catboost'           # 'catboost', 'xgboost', 'lgbm'

# Optuna
optuna_n_trials: int = 20
optuna_timeout_s: int | None = None
optimize_metric: str = 'f1'            # 'f1', 'precision', 'recall'
threshold_grid: list[float] = [round(x * 0.05, 2) for x in range(1, 20)]

# SHAP
enable_shap: bool = True
shap_sample_size: int = 500

# Outputs
metrics_json_path: str = 'outputs/metrics/metrics.json'
model_output_path: str = 'outputs/models/model_best.cbm'
plots_dir: str = 'outputs/plots'
executed_notebook_path: str | None = None

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import json
import os
import warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    roc_curve,
)

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN

import shap

# ---------------------------------------------------------------------------
# Create output directories
# ---------------------------------------------------------------------------
for _d in [plots_dir, 'outputs/runs', 'outputs/models', 'outputs/metrics']:
    Path(_d).mkdir(parents=True, exist_ok=True)

RUN_START = datetime.now(timezone.utc)
print(f'Run started at {RUN_START.isoformat()}')
print(f'base_model={base_model!r}, optimize_metric={optimize_metric!r}, optuna_n_trials={optuna_n_trials}')

## 1 — Data Loading

In [ ]:
# ---------------------------------------------------------------------------
# Data Loading
# ---------------------------------------------------------------------------

def load_data() -> pl.DataFrame:
    if not feature_paths:
        print(f'No feature_paths provided — generating synthetic dataset.')
        print(f'  n_samples=5000, n_features=20, imbalance_ratio={imbalance_ratio}')
        X_raw, y_raw = make_classification(
            n_samples=5000,
            n_features=20,
            n_informative=12,
            n_redundant=4,
            n_classes=2,
            weights=[1 - imbalance_ratio, imbalance_ratio],
            flip_y=0.01,
            random_state=random_state,
        )
        feature_cols = {f'feature_{i:02d}': X_raw[:, i] for i in range(X_raw.shape[1])}
        feature_cols[target_col] = y_raw.astype(int)
        return pl.DataFrame(feature_cols)

    dfs = []
    for path in feature_paths:
        if path.endswith('.parquet'):
            dfs.append(pl.read_parquet(path))
        elif path.endswith('.csv'):
            dfs.append(pl.read_csv(path))
        else:
            raise ValueError(f'Unsupported file format: {path}')
    return pl.concat(dfs)


df = load_data()

# Recode positive_label if specified
if positive_label is not None:
    df = df.with_columns(
        pl.when(pl.col(target_col) == positive_label)
        .then(1)
        .otherwise(0)
        .alias(target_col)
    )

print(f'\nDataset shape: {df.shape}')
print(f'\nClass distribution:')
value_counts = df[target_col].value_counts().sort(target_col)
print(value_counts)

n_pos = int((df[target_col] == 1).sum())
n_neg = int((df[target_col] == 0).sum())
n_total = len(df)
actual_ratio = n_pos / n_total
print(f'\nPositive class ratio: {actual_ratio:.4f} ({n_pos}/{n_total})')
print(f'Imbalance ratio (neg:pos): {n_neg/n_pos:.1f}:1')

## 2 — EDA

In [ ]:
# ---------------------------------------------------------------------------
# EDA
# ---------------------------------------------------------------------------

feature_cols = [c for c in df.columns if c != target_col]
print(f'Features: {len(feature_cols)}')
print(f'\nClass distribution (counts):')
for val, cnt in value_counts.iter_rows():
    pct = cnt / n_total * 100
    print(f'  class {val}: {cnt:,} ({pct:.2f}%)')
print(f'\nImbalance ratio: {n_neg/n_pos:.2f}:1 (neg:pos)')

# Class distribution bar chart
fig, ax = plt.subplots(figsize=(5, 4))
classes = [str(r[0]) for r in value_counts.iter_rows()]
counts = [r[1] for r in value_counts.iter_rows()]
colors = ['#2196F3', '#F44336']
ax.bar(classes, counts, color=colors[:len(classes)])
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.set_title('Class Distribution')
for i, (c, v) in enumerate(zip(classes, counts)):
    ax.text(i, v + n_total * 0.005, f'{v:,}\n({v/n_total*100:.1f}%)',
            ha='center', va='bottom', fontsize=9)
plt.tight_layout()
eda_class_path = os.path.join(plots_dir, 'eda_class_dist.png')
fig.savefig(eda_class_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {eda_class_path}')

# Feature histograms (first 8)
plot_cols = feature_cols[:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
df_pd = df.to_pandas()
for i, col in enumerate(plot_cols):
    ax = axes[i // 4][i % 4]
    ax.hist(df_pd[df_pd[target_col] == 0][col], bins=30, alpha=0.6, label='class 0', color='#2196F3')
    ax.hist(df_pd[df_pd[target_col] == 1][col], bins=30, alpha=0.6, label='class 1', color='#F44336')
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)
plt.suptitle('Feature Distributions by Class (first 8 features)', fontsize=12)
plt.tight_layout()
eda_feat_path = os.path.join(plots_dir, 'eda_features.png')
fig.savefig(eda_feat_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {eda_feat_path}')

## 3 — Train / Test Split

In [ ]:
# ---------------------------------------------------------------------------
# Train / Test Split
# ---------------------------------------------------------------------------

df_pd = df.to_pandas()
X = df_pd[feature_cols].values.astype(np.float32)
y = df_pd[target_col].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=test_size,
    random_state=random_state,
    stratify=y,
)

n_train_pos = int(y_train.sum())
n_train_neg = int((y_train == 0).sum())
n_test_pos = int(y_test.sum())
n_test_neg = int((y_test == 0).sum())
scale_pos_weight = n_train_neg / max(n_train_pos, 1)

print(f'Train size: {len(X_train):,}  (pos={n_train_pos:,}, neg={n_train_neg:,}, rate={n_train_pos/len(y_train):.4f})')
print(f'Test  size: {len(X_test):,}  (pos={n_test_pos:,}, neg={n_test_neg:,}, rate={n_test_pos/len(y_test):.4f})')
print(f'scale_pos_weight (neg/pos): {scale_pos_weight:.2f}')

## 4 — Strategy Comparison

In [ ]:
# ---------------------------------------------------------------------------
# Model factory + strategy definitions
# ---------------------------------------------------------------------------

def get_base_model(strategy_name: str, use_class_weight: bool):
    """Return a fresh untrained model appropriate for the strategy."""
    spw = scale_pos_weight if use_class_weight else 1.0
    if base_model == 'catboost':
        return CatBoostClassifier(
            iterations=500,
            scale_pos_weight=spw,
            random_seed=random_state,
            verbose=0,
        )
    elif base_model == 'xgboost':
        return XGBClassifier(
            n_estimators=500,
            scale_pos_weight=spw,
            random_state=random_state,
            verbosity=0,
            eval_metric='logloss',
        )
    elif base_model == 'lgbm':
        cw = 'balanced' if use_class_weight else None
        return LGBMClassifier(
            n_estimators=500,
            class_weight=cw,
            random_state=random_state,
            verbose=-1,
        )
    else:
        raise ValueError(f'Unknown base_model: {base_model!r}. Choose catboost, xgboost, or lgbm.')


# Strategy registry: name -> (sampler_or_None, use_class_weight)
strategies = {
    'none':        (None, False),
    'class_weight': (None, True),
    'smote':       (SMOTE(random_state=random_state), False),
    'adasyn':      (ADASYN(random_state=random_state), False),
    'undersample': (RandomUnderSampler(random_state=random_state), False),
    'smote_enn':   (SMOTEENN(random_state=random_state), False),
}

METRIC_FN = {
    'f1':        lambda yt, yp: f1_score(yt, yp, zero_division=0),
    'precision': lambda yt, yp: precision_score(yt, yp, zero_division=0),
    'recall':    lambda yt, yp: recall_score(yt, yp, zero_division=0),
}

print(f'Strategies to evaluate: {list(strategies.keys())}')
print(f'Optimize metric: {optimize_metric!r}')

In [ ]:
# ---------------------------------------------------------------------------
# Train all strategies and collect results
# ---------------------------------------------------------------------------

def find_best_threshold(model, X_val, y_val):
    """Find threshold that maximises optimize_metric on a validation fold."""
    y_prob = model.predict_proba(X_val)[:, 1]
    metric_fn = METRIC_FN[optimize_metric]
    best_thr = 0.5
    best_score = -1.0
    for thr in threshold_grid:
        y_pred = (y_prob >= thr).astype(int)
        score = metric_fn(y_val, y_pred)
        if score > best_score:
            best_score = score
            best_thr = thr
    return best_thr


def evaluate_on_test(model, X_te, y_te, threshold):
    """Return dict of test metrics at given threshold."""
    y_prob = model.predict_proba(X_te)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'precision': float(precision_score(y_te, y_pred, zero_division=0)),
        'recall':    float(recall_score(y_te, y_pred, zero_division=0)),
        'f1':        float(f1_score(y_te, y_pred, zero_division=0)),
        'roc_auc':   float(roc_auc_score(y_te, y_prob)),
        'pr_auc':    float(average_precision_score(y_te, y_prob)),
        'threshold': float(threshold),
        'y_prob':    y_prob,
    }


all_results = {}
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)

for strat_name, (sampler, use_cw) in strategies.items():
    print(f'\n--- Strategy: {strat_name} ---')

    # Apply resampling
    if sampler is not None:
        try:
            X_res, y_res = sampler.fit_resample(X_train, y_train)
            print(f'  Resampled: {len(X_res):,} samples  (pos={int(y_res.sum()):,}, neg={int((y_res==0).sum()):,})')
        except Exception as e:
            print(f'  Resampling failed ({e}), falling back to original data')
            X_res, y_res = X_train.copy(), y_train.copy()
    else:
        X_res, y_res = X_train.copy(), y_train.copy()

    # CV threshold search
    fold_thresholds = []
    for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X_res, y_res)):
        X_tr_f, X_val_f = X_res[tr_idx], X_res[val_idx]
        y_tr_f, y_val_f = y_res[tr_idx], y_res[val_idx]
        m = get_base_model(strat_name, use_cw)
        m.fit(X_tr_f, y_tr_f)
        thr = find_best_threshold(m, X_val_f, y_val_f)
        fold_thresholds.append(thr)
        print(f'  Fold {fold_idx+1}: best threshold = {thr:.2f}')

    best_thr = float(np.median(fold_thresholds))
    print(f'  Median CV threshold: {best_thr:.2f}')

    # Final model on full resampled train
    final_model = get_base_model(strat_name, use_cw)
    final_model.fit(X_res, y_res)

    # Evaluate on test
    metrics = evaluate_on_test(final_model, X_test, y_test, best_thr)
    metrics['model'] = final_model
    all_results[strat_name] = metrics

    print(f'  Test  PR-AUC={metrics["pr_auc"]:.4f}  ROC-AUC={metrics["roc_auc"]:.4f}  '
          f'F1={metrics["f1"]:.4f}  P={metrics["precision"]:.4f}  R={metrics["recall"]:.4f}')

print('\n' + '='*70)
print(f'{"Strategy":<15} {"PR-AUC":>8} {"ROC-AUC":>8} {"F1":>8} {"Precision":>10} {"Recall":>8} {"Thr":>6}')
print('-'*70)
for name, res in all_results.items():
    print(f'{name:<15} {res["pr_auc"]:>8.4f} {res["roc_auc"]:>8.4f} '
          f'{res["f1"]:>8.4f} {res["precision"]:>10.4f} {res["recall"]:>8.4f} {res["threshold"]:>6.2f}')

In [ ]:
# ---------------------------------------------------------------------------
# PR-AUC comparison bar chart
# ---------------------------------------------------------------------------

strat_names = list(all_results.keys())
pr_aucs = [all_results[s]['pr_auc'] for s in strat_names]
best_idx = int(np.argmax(pr_aucs))

fig, ax = plt.subplots(figsize=(9, 5))
colors_bar = ['#FF6B35' if i == best_idx else '#2196F3' for i in range(len(strat_names))]
bars = ax.bar(strat_names, pr_aucs, color=colors_bar, edgecolor='white', linewidth=0.8)
ax.set_xlabel('Strategy')
ax.set_ylabel('PR-AUC')
ax.set_title('PR-AUC by Imbalance Strategy')
ax.set_ylim(0, min(1.05, max(pr_aucs) * 1.15))
for bar, val in zip(bars, pr_aucs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax.legend(handles=[
    plt.Rectangle((0,0), 1, 1, color='#FF6B35', label='Best'),
    plt.Rectangle((0,0), 1, 1, color='#2196F3', label='Other'),
], loc='lower right')
plt.xticks(rotation=15)
plt.tight_layout()
path = os.path.join(plots_dir, 'strategy_pr_auc.png')
fig.savefig(path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

In [ ]:
# ---------------------------------------------------------------------------
# F1 and ROC-AUC comparison side by side
# ---------------------------------------------------------------------------

f1_scores = [all_results[s]['f1'] for s in strat_names]
roc_aucs  = [all_results[s]['roc_auc'] for s in strat_names]
best_f1_idx  = int(np.argmax(f1_scores))
best_roc_idx = int(np.argmax(roc_aucs))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1
ax = axes[0]
cols = ['#FF6B35' if i == best_f1_idx else '#2196F3' for i in range(len(strat_names))]
bars = ax.bar(strat_names, f1_scores, color=cols, edgecolor='white')
ax.set_title('F1 Score by Strategy')
ax.set_ylabel('F1')
ax.set_ylim(0, min(1.05, max(f1_scores) * 1.2))
ax.set_xticklabels(strat_names, rotation=15)
for bar, val in zip(bars, f1_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8)

# ROC-AUC
ax = axes[1]
cols = ['#FF6B35' if i == best_roc_idx else '#4CAF50' for i in range(len(strat_names))]
bars = ax.bar(strat_names, roc_aucs, color=cols, edgecolor='white')
ax.set_title('ROC-AUC by Strategy')
ax.set_ylabel('ROC-AUC')
ax.set_ylim(0, min(1.05, max(roc_aucs) * 1.05))
ax.set_xticklabels(strat_names, rotation=15)
for bar, val in zip(bars, roc_aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
path = os.path.join(plots_dir, 'strategy_metrics.png')
fig.savefig(path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

In [ ]:
# ---------------------------------------------------------------------------
# Precision-Recall curves — all strategies on one plot
# ---------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 6))
palette = plt.cm.tab10.colors

for i, strat_name in enumerate(strat_names):
    y_prob = all_results[strat_name]['y_prob']
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = all_results[strat_name]['pr_auc']
    lw = 2.5 if i == best_idx else 1.0
    ls = '-' if i == best_idx else '--'
    ax.plot(rec, prec, color=palette[i % 10], lw=lw, ls=ls,
            label=f'{strat_name} (PR-AUC={pr_auc:.4f})')

baseline = n_test_pos / len(y_test)
ax.axhline(baseline, color='gray', lw=1, ls=':', label=f'Baseline (random) = {baseline:.4f}')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves — All Strategies')
ax.legend(loc='upper right', fontsize=8)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
plt.tight_layout()
path = os.path.join(plots_dir, 'pr_curves_all_strategies.png')
fig.savefig(path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

In [ ]:
# ---------------------------------------------------------------------------
# Confusion matrices — 2x3 grid
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes_flat = axes.flatten()

for i, strat_name in enumerate(strat_names):
    y_prob = all_results[strat_name]['y_prob']
    thr = all_results[strat_name]['threshold']
    y_pred = (y_prob >= thr).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Neg', 'Pos'])
    disp.plot(ax=axes_flat[i], colorbar=False, cmap='Blues')
    f1_val = all_results[strat_name]['f1']
    pr_val = all_results[strat_name]['pr_auc']
    axes_flat[i].set_title(f'{strat_name}\nF1={f1_val:.4f}  PR-AUC={pr_val:.4f}', fontsize=9)

# Hide any unused axes
for j in range(len(strat_names), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Confusion Matrices by Strategy', fontsize=13, y=1.01)
plt.tight_layout()
path = os.path.join(plots_dir, 'confusion_matrices.png')
fig.savefig(path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

In [ ]:
# ---------------------------------------------------------------------------
# Identify best strategy by PR-AUC
# ---------------------------------------------------------------------------

best_strategy = strat_names[best_idx]
best_metrics  = all_results[best_strategy]

print(f'Best strategy by PR-AUC: {best_strategy!r}')
print(f'  PR-AUC    : {best_metrics["pr_auc"]:.4f}')
print(f'  ROC-AUC   : {best_metrics["roc_auc"]:.4f}')
print(f'  F1        : {best_metrics["f1"]:.4f}')
print(f'  Precision : {best_metrics["precision"]:.4f}')
print(f'  Recall    : {best_metrics["recall"]:.4f}')
print(f'  Threshold : {best_metrics["threshold"]:.2f}')

## 5 — Best Strategy: Deep Dive

In [ ]:
# ---------------------------------------------------------------------------
# Optuna tuning for best strategy
# ---------------------------------------------------------------------------

best_sampler, best_use_cw = strategies[best_strategy]

# Resample with best strategy
if best_sampler is not None:
    try:
        X_res_best, y_res_best = best_sampler.fit_resample(X_train, y_train)
        print(f'Resampled with {best_strategy}: {len(X_res_best):,} samples')
    except Exception as e:
        print(f'Resampling failed ({e}), using original data')
        X_res_best, y_res_best = X_train.copy(), y_train.copy()
else:
    X_res_best, y_res_best = X_train.copy(), y_train.copy()
    print(f'Strategy {best_strategy!r} uses no resampler.')

spw_best = scale_pos_weight if best_use_cw else 1.0
metric_fn = METRIC_FN[optimize_metric]

def objective(trial: optuna.Trial) -> float:
    if base_model == 'catboost':
        params = dict(
            iterations=trial.suggest_int('iterations', 200, 1500),
            depth=trial.suggest_int('depth', 3, 10),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 0.01, 10.0, log=True),
            scale_pos_weight=spw_best,
            random_seed=random_state,
            verbose=0,
        )
        model_cls = CatBoostClassifier
    elif base_model == 'xgboost':
        params = dict(
            n_estimators=trial.suggest_int('n_estimators', 100, 1500),
            max_depth=trial.suggest_int('max_depth', 3, 10),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            reg_lambda=trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
            scale_pos_weight=spw_best,
            random_state=random_state,
            verbosity=0,
            eval_metric='logloss',
        )
        model_cls = XGBClassifier
    else:  # lgbm
        cw = 'balanced' if best_use_cw else None
        params = dict(
            n_estimators=trial.suggest_int('n_estimators', 100, 1500),
            max_depth=trial.suggest_int('max_depth', 3, 10),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            reg_lambda=trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
            class_weight=cw,
            random_state=random_state,
            verbose=-1,
        )
        model_cls = LGBMClassifier

    cv_scores = []
    for tr_idx, val_idx in StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state).split(X_res_best, y_res_best):
        m = model_cls(**params)
        m.fit(X_res_best[tr_idx], y_res_best[tr_idx])
        y_prob_val = m.predict_proba(X_res_best[val_idx])[:, 1]
        # find best threshold on val
        best_s = 0.0
        for thr in threshold_grid:
            s = metric_fn(y_res_best[val_idx], (y_prob_val >= thr).astype(int))
            if s > best_s:
                best_s = s
        cv_scores.append(best_s)
    return float(np.mean(cv_scores))


print(f'\nRunning Optuna with {optuna_n_trials} trials on strategy: {best_strategy!r} ...')
study = optuna.create_study(direction='maximize')
study.optimize(
    objective,
    n_trials=optuna_n_trials,
    timeout=optuna_timeout_s,
    show_progress_bar=False,
)

print(f'Best trial value ({optimize_metric}): {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

# Retrain final tuned model on full resampled train
best_params = study.best_params.copy()
if base_model == 'catboost':
    best_params.update({'scale_pos_weight': spw_best, 'random_seed': random_state, 'verbose': 0})
    tuned_model = CatBoostClassifier(**best_params)
elif base_model == 'xgboost':
    best_params.update({'scale_pos_weight': spw_best, 'random_state': random_state, 'verbosity': 0})
    tuned_model = XGBClassifier(**best_params)
else:
    cw = 'balanced' if best_use_cw else None
    best_params.update({'class_weight': cw, 'random_state': random_state, 'verbose': -1})
    tuned_model = LGBMClassifier(**best_params)

tuned_model.fit(X_res_best, y_res_best)
print('\nTuned model trained on full resampled train set.')

# Find best threshold on tuned model
tuned_thresholds = []
for tr_idx, val_idx in StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state).split(X_res_best, y_res_best):
    m_tmp = tuned_model.__class__(**{k: v for k, v in best_params.items()})
    # use the already-tuned model for threshold only (fit on fold)
    m_tmp2 = tuned_model.__class__(**best_params) if base_model != 'catboost' else CatBoostClassifier(**best_params)
    m_tmp2.fit(X_res_best[tr_idx], y_res_best[tr_idx])
    tuned_thresholds.append(find_best_threshold(m_tmp2, X_res_best[val_idx], y_res_best[val_idx]))

tuned_threshold = float(np.median(tuned_thresholds))
print(f'Tuned threshold (median CV): {tuned_threshold:.2f}')

# Save model
Path(model_output_path).parent.mkdir(parents=True, exist_ok=True)
if base_model == 'catboost':
    tuned_model.save_model(model_output_path)
    print(f'Model saved to {model_output_path}')
elif base_model == 'xgboost':
    xgb_path = model_output_path.replace('.cbm', '.json')
    tuned_model.save_model(xgb_path)
    model_output_path = xgb_path
    print(f'Model saved to {model_output_path}')
else:
    import joblib
    lgbm_path = model_output_path.replace('.cbm', '.pkl')
    joblib.dump(tuned_model, lgbm_path)
    model_output_path = lgbm_path
    print(f'Model saved to {model_output_path}')

# Evaluate tuned model
tuned_metrics = evaluate_on_test(tuned_model, X_test, y_test, tuned_threshold)
print(f'\nTuned model test metrics:')
print(f'  PR-AUC    : {tuned_metrics["pr_auc"]:.4f}')
print(f'  ROC-AUC   : {tuned_metrics["roc_auc"]:.4f}')
print(f'  F1        : {tuned_metrics["f1"]:.4f}')
print(f'  Precision : {tuned_metrics["precision"]:.4f}')
print(f'  Recall    : {tuned_metrics["recall"]:.4f}')

In [ ]:
# ---------------------------------------------------------------------------
# Threshold analysis for tuned best model
# ---------------------------------------------------------------------------

y_prob_tuned = tuned_model.predict_proba(X_test)[:, 1]

th_precisions = []
th_recalls = []
th_f1s = []

for thr in threshold_grid:
    y_pred_thr = (y_prob_tuned >= thr).astype(int)
    th_precisions.append(precision_score(y_test, y_pred_thr, zero_division=0))
    th_recalls.append(recall_score(y_test, y_pred_thr, zero_division=0))
    th_f1s.append(f1_score(y_test, y_pred_thr, zero_division=0))

best_f1_thr  = threshold_grid[int(np.argmax(th_f1s))]
best_p_thr   = threshold_grid[int(np.argmax(th_precisions))]
best_r_thr   = threshold_grid[int(np.argmax(th_recalls))]

print(f'Best threshold for F1       : {best_f1_thr:.2f}  (F1={max(th_f1s):.4f})')
print(f'Best threshold for Precision: {best_p_thr:.2f}  (P={max(th_precisions):.4f})')
print(f'Best threshold for Recall   : {best_r_thr:.2f}  (R={max(th_recalls):.4f})')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(threshold_grid, th_precisions, 'b-o', ms=4, label='Precision')
ax.plot(threshold_grid, th_recalls,    'r-o', ms=4, label='Recall')
ax.plot(threshold_grid, th_f1s,        'g-o', ms=4, label='F1')
ax.axvline(tuned_threshold, color='gray', ls='--', lw=1.2, label=f'Selected thr={tuned_threshold:.2f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title(f'Threshold Analysis — {best_strategy} (tuned {base_model})')
ax.legend()
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.grid(alpha=0.3)
plt.tight_layout()
path = os.path.join(plots_dir, 'best_threshold_analysis.png')
fig.savefig(path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

In [ ]:
# ---------------------------------------------------------------------------
# ROC and PR curves for best tuned model
# ---------------------------------------------------------------------------

fpr, tpr, _ = roc_curve(y_test, y_prob_tuned)
prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_prob_tuned)
roc_auc_val = tuned_metrics['roc_auc']
pr_auc_val  = tuned_metrics['pr_auc']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(fpr, tpr, color='#2196F3', lw=2, label=f'ROC-AUC = {roc_auc_val:.4f}')
ax.plot([0, 1], [0, 1], color='gray', lw=1, ls='--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curve — {best_strategy} (tuned)')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

ax = axes[1]
ax.plot(rec_curve, prec_curve, color='#FF6B35', lw=2, label=f'PR-AUC = {pr_auc_val:.4f}')
baseline = n_test_pos / len(y_test)
ax.axhline(baseline, color='gray', lw=1, ls='--', label=f'Baseline = {baseline:.4f}')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f'PR Curve — {best_strategy} (tuned)')
ax.legend(loc='upper right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])

plt.suptitle(f'Best Strategy: {best_strategy} | {base_model} (tuned)', fontsize=12)
plt.tight_layout()
path = os.path.join(plots_dir, 'best_roc_pr_curves.png')
fig.savefig(path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

In [ ]:
# ---------------------------------------------------------------------------
# SHAP analysis for best tuned model
# ---------------------------------------------------------------------------

if enable_shap:
    print(f'Computing SHAP values (sample_size={shap_sample_size}) ...')
    shap_sample_size_actual = min(shap_sample_size, len(X_test))
    rng = np.random.default_rng(random_state)
    shap_idx = rng.choice(len(X_test), size=shap_sample_size_actual, replace=False)
    X_shap = X_test[shap_idx]

    try:
        explainer = shap.TreeExplainer(tuned_model)
        shap_values = explainer.shap_values(X_shap)

        # Handle list output (binary classification returns list of 2 arrays for some models)
        if isinstance(shap_values, list):
            shap_vals_plot = shap_values[1]  # positive class
        else:
            shap_vals_plot = shap_values

        # If 3D array (some lgbm configs), take last slice
        if shap_vals_plot.ndim == 3:
            shap_vals_plot = shap_vals_plot[:, :, 1]

        feat_names = [f'feature_{i:02d}' for i in range(X_shap.shape[1])]

        # Bar summary
        fig, ax = plt.subplots(figsize=(8, 6))
        shap.summary_plot(shap_vals_plot, X_shap, feature_names=feat_names,
                          plot_type='bar', show=False)
        plt.title(f'SHAP Feature Importance — {best_strategy} ({base_model})')
        plt.tight_layout()
        path_bar = os.path.join(plots_dir, 'shap_bar.png')
        plt.savefig(path_bar, dpi=120, bbox_inches='tight')
        plt.show()
        print(f'Saved: {path_bar}')

        # Dot summary
        fig, ax = plt.subplots(figsize=(8, 7))
        shap.summary_plot(shap_vals_plot, X_shap, feature_names=feat_names,
                          plot_type='dot', show=False)
        plt.title(f'SHAP Summary — {best_strategy} ({base_model})')
        plt.tight_layout()
        path_dot = os.path.join(plots_dir, 'shap_summary.png')
        plt.savefig(path_dot, dpi=120, bbox_inches='tight')
        plt.show()
        print(f'Saved: {path_dot}')

    except Exception as e:
        print(f'SHAP computation failed: {e}')
else:
    print('SHAP disabled (enable_shap=False)')

## 6.1 — Cost-Sensitive Decision Making

PR-AUC is a useful aggregate metric for imbalanced problems, but real 
business decisions involve **asymmetric costs**:
- **False Negative cost:** missing a fraud → €500 average loss
- **False Positive cost:** wrongly flagging a legitimate transaction → €5 investigation cost

With a cost matrix, the **optimal threshold** shifts dramatically. Instead 
of maximizing F1 (which equally weights precision and recall), we minimize 
**expected cost per prediction**.

Expected cost at threshold t:
```
EC(t) = FP_rate(t) × cost_FP × P(negative) + FN_rate(t) × cost_FN × P(positive)
```

The **cost-optimal threshold** often differs significantly from the F1-optimal 
threshold — and the choice can have a large impact on business outcomes. 
This analysis is *the* key skill for deploying classifiers in production.

In [ ]:
# ---------------------------------------------------------------------------
# Cost-Sensitive Threshold Optimization
# ---------------------------------------------------------------------------
from sklearn.metrics import precision_recall_curve as _prc

# Define cost matrix (configurable)
cost_fn = 20.0   # cost of a False Negative (missing positive = expensive)
cost_fp = 1.0    # cost of a False Positive (false alarm = cheap)
prevalence = float(y_test.mean())

print(f"Cost matrix:")
print(f"  False Negative cost : {cost_fn:.1f}x")
print(f"  False Positive cost : {cost_fp:.1f}x")
print(f"  Cost ratio (FN/FP)  : {cost_fn / cost_fp:.1f}x")
print(f"  Prevalence (% pos)  : {100 * prevalence:.2f}%")
print()

# Use the best tuned model's predictions
y_proba_best = best_tuned_model.predict_proba(X_test)[:, 1]

# Sweep thresholds
thresholds_sweep = np.linspace(0.01, 0.99, 200)
costs_per_threshold = []
f1_per_threshold = []

for thr in thresholds_sweep:
    y_pred_t = (y_proba_best >= thr).astype(int)
    tn = int(((y_pred_t == 0) & (y_test == 0)).sum())
    fp = int(((y_pred_t == 1) & (y_test == 0)).sum())
    fn = int(((y_pred_t == 0) & (y_test == 1)).sum())
    tp = int(((y_pred_t == 1) & (y_test == 1)).sum())
    n = len(y_test)

    # Expected cost per sample
    fp_rate = fp / max(tn + fp, 1)
    fn_rate = fn / max(tp + fn, 1)
    expected_cost = fp_rate * cost_fp * (1 - prevalence) + fn_rate * cost_fn * prevalence
    costs_per_threshold.append(expected_cost)
    f1_per_threshold.append(float(f1_score(y_test, y_pred_t, zero_division=0)))

costs_arr = np.array(costs_per_threshold)
f1_arr    = np.array(f1_per_threshold)

cost_optimal_thr = thresholds_sweep[np.argmin(costs_arr)]
f1_optimal_thr   = thresholds_sweep[np.argmax(f1_arr)]

print(f"F1-optimal threshold  : {f1_optimal_thr:.3f}  → F1={f1_arr.max():.4f},  cost={costs_arr[np.argmax(f1_arr)]:.4f}")
print(f"Cost-optimal threshold: {cost_optimal_thr:.3f}  → cost={costs_arr.min():.4f},  F1={f1_arr[np.argmin(costs_arr)]:.4f}")
print()

# Compute metrics at both thresholds
for label, thr in [("F1-optimal", f1_optimal_thr), ("Cost-optimal", cost_optimal_thr)]:
    y_pred_t = (y_proba_best >= thr).astype(int)
    prec = float(precision_score(y_test, y_pred_t, zero_division=0))
    rec  = float(recall_score(y_test, y_pred_t, zero_division=0))
    f1   = float(f1_score(y_test, y_pred_t, zero_division=0))
    print(f"{label} (thr={thr:.3f}): precision={prec:.3f}, recall={rec:.3f}, F1={f1:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Expected cost vs threshold
ax = axes[0]
ax.plot(thresholds_sweep, costs_arr, color='steelblue', linewidth=2)
ax.axvline(cost_optimal_thr, color='tomato', linestyle='--', linewidth=2,
           label=f'Cost-optimal thr={cost_optimal_thr:.3f}')
ax.axvline(f1_optimal_thr, color='darkorange', linestyle='--', linewidth=2,
           label=f'F1-optimal thr={f1_optimal_thr:.3f}')
ax.scatter([cost_optimal_thr], [costs_arr.min()], color='tomato', s=100, zorder=5)
ax.set_xlabel('Decision Threshold', fontsize=11)
ax.set_ylabel(f'Expected Cost (FN\u00d7{cost_fn:.0f} + FP\u00d7{cost_fp:.0f})', fontsize=11)
ax.set_title(f'Expected Cost vs Threshold\n(asymmetric costs: FN={cost_fn:.0f}\u00d7, FP={cost_fp:.0f}\u00d7)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 2: F1 and cost on same x-axis to show divergence
ax = axes[1]
ax_r = ax.twinx()
ax.plot(thresholds_sweep, f1_arr, color='steelblue', linewidth=2, label='F1')
ax_r.plot(thresholds_sweep, costs_arr, color='tomato', linewidth=2, linestyle='--',
          label='Expected Cost')
ax.axvline(cost_optimal_thr, color='tomato', linestyle=':', linewidth=1.5)
ax.axvline(f1_optimal_thr, color='steelblue', linestyle=':', linewidth=1.5)
ax.set_xlabel('Decision Threshold', fontsize=11)
ax.set_ylabel('F1 Score', fontsize=10, color='steelblue')
ax_r.set_ylabel('Expected Cost', fontsize=10, color='tomato')
ax.set_title(f'F1 vs Expected Cost — Threshold Divergence\n'
             f'Gap = {abs(cost_optimal_thr - f1_optimal_thr):.3f}', fontsize=11)
ax.legend(loc='lower left', fontsize=9)
ax_r.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
_cost_path = os.path.join(plots_dir, 'cost_sensitive_threshold.png')
fig.savefig(_cost_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {_cost_path}')
print('\nKey insight: the F1-optimal threshold maximizes balanced precision/recall.')
print('The cost-optimal threshold shifts to recall more positives when FN cost >> FP cost.')

## 6.2 — SMOTE: What Synthetic Samples Actually Look Like

SMOTE (Synthetic Minority Oversampling TEchnique) generates **synthetic** 
minority-class samples by interpolating between existing minority points:

1. Pick a minority sample `x`
2. Find its k nearest minority neighbors
3. Pick one neighbor `x_n` at random
4. Create a new sample: `x_new = x + α × (x_n - x)` where α ∼ Uniform(0, 1)

This means SMOTE **fills the feature space** between real minority samples 
— it doesn't duplicate them. But it has failure modes:

- **Overlap regions:** if minority and majority samples are interleaved, 
  SMOTE creates synthetic samples in majority territory → noisy training
- **High-cardinality categorical features:** interpolation doesn't preserve 
  valid category values
- **Cluster boundaries:** SMOTE doesn't respect cluster structure — 
  it can create samples spanning multiple minority clusters

Visualizing SMOTE in 2D (via PCA) shows both its strengths and failure modes.

In [ ]:
# ---------------------------------------------------------------------------
# SMOTE Visualization — Before and After in 2D PCA Space
# ---------------------------------------------------------------------------
from imblearn.over_sampling import SMOTE as _SMOTE
from sklearn.decomposition import PCA as _PCA

# Apply SMOTE to training data
smote_viz = _SMOTE(random_state=random_state)
try:
    X_smote, y_smote = smote_viz.fit_resample(X_train, y_train)
except Exception as e:
    print(f"SMOTE failed: {e}")
    X_smote, y_smote = X_train.copy(), y_train.copy()

n_orig_pos = int((y_train == 1).sum())
n_orig_neg = int((y_train == 0).sum())
n_synth    = int((y_smote == 1).sum()) - n_orig_pos
print(f"Before SMOTE: {n_orig_neg} negatives, {n_orig_pos} positives")
print(f"After SMOTE : {int((y_smote == 0).sum())} negatives, {int((y_smote == 1).sum())} positives")
print(f"Synthetic minority samples created: {n_synth}")

# PCA for 2D visualization
pca_smote = _PCA(n_components=2, random_state=random_state)
# Fit on combined data
X_all_smote = np.vstack([X_smote, X_train])  # include both for stable PCA
pca_smote.fit(X_all_smote)

X_train_2d_pre  = pca_smote.transform(X_train[y_train == 1])
X_train_2d_neg  = pca_smote.transform(X_train[y_train == 0])
X_smote_2d_all  = pca_smote.transform(X_smote)

# Separate original and synthetic minority samples
n_orig = len(X_train)
X_smote_orig = X_smote_2d_all[y_smote == 1][:n_orig_pos]  # original minority
X_smote_synth = X_smote_2d_all[y_smote == 1][n_orig_pos:]  # synthetic minority

exp_var = pca_smote.explained_variance_ratio_
print(f"\nPCA explained variance: {100*exp_var.sum():.1f}% ({exp_var[0]:.1%} + {exp_var[1]:.1%})")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scatter_kw = dict(alpha=0.5, linewidths=0)

# Plot 1: Before SMOTE
ax = axes[0]
X_train_2d = pca_smote.transform(X_train)
ax.scatter(X_train_2d[y_train == 0, 0], X_train_2d[y_train == 0, 1],
           c='steelblue', s=8, label=f'Majority (n={n_orig_neg})', **scatter_kw)
ax.scatter(X_train_2d[y_train == 1, 0], X_train_2d[y_train == 1, 1],
           c='tomato', s=30, label=f'Minority (n={n_orig_pos})', marker='^', **scatter_kw)
ax.set_title(f'Before SMOTE\nClass ratio: {n_orig_neg}:{n_orig_pos} = {n_orig_neg/max(n_orig_pos,1):.1f}:1', fontsize=11)
ax.set_xlabel(f'PCA-1 ({exp_var[0]:.1%} var)')
ax.set_ylabel(f'PCA-2 ({exp_var[1]:.1%} var)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# Plot 2: After SMOTE — highlight synthetic vs real minority
ax = axes[1]
ax.scatter(X_train_2d[y_train == 0, 0], X_train_2d[y_train == 0, 1],
           c='steelblue', s=8, label=f'Majority (n={n_orig_neg})', **scatter_kw)
ax.scatter(X_train_2d[y_train == 1, 0], X_train_2d[y_train == 1, 1],
           c='tomato', s=40, label=f'Real minority (n={n_orig_pos})', marker='^',
           alpha=0.9, linewidths=0.5, edgecolors='darkred', zorder=4)
if len(X_smote_synth) > 0:
    ax.scatter(X_smote_synth[:, 0], X_smote_synth[:, 1],
               c='orange', s=20, label=f'SMOTE synthetic (n={len(X_smote_synth)})',
               marker='o', alpha=0.4, linewidths=0)
ax.set_title(f'After SMOTE\nSynthetic samples fill minority region', fontsize=11)
ax.set_xlabel(f'PCA-1 ({exp_var[0]:.1%} var)')
ax.set_ylabel(f'PCA-2 ({exp_var[1]:.1%} var)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

plt.tight_layout()
_smote_path = os.path.join(plots_dir, 'smote_visualization.png')
fig.savefig(_smote_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {_smote_path}')
print('\nNote: Synthetic samples appear between existing minority samples (interpolation).')
print('If synthetic samples overlap with the majority class, SMOTE may harm performance.')

## 6.3 — Strategy Comparison: Precision-Recall Tradeoff Curves

Each sampling strategy shifts the **precision-recall tradeoff** differently.
Rather than comparing strategies at a single threshold, the **full PR curve** 
reveals which strategy achieves better precision at any given recall level.

The **area under the PR curve (PR-AUC)** summarizes this — but the *shape* 
of the curve matters too:
- A strategy with high **early precision** (top-left of PR curve) is best 
  when you need high precision at low recall budgets (e.g., only review 
  the top 5% of flagged cases)
- A strategy with a **long tail** maintains recall at the cost of precision 
  (better when missing positives is very costly)

The crossover points between PR curves reveal which strategy is superior 
in different operating regimes — information that a single PR-AUC number hides.

In [ ]:
# ---------------------------------------------------------------------------
# Full PR Curves Comparison — All Strategies on the Best Model Type
# ---------------------------------------------------------------------------
from sklearn.metrics import precision_recall_curve as _prc2, average_precision_score as _ap

# Re-evaluate all strategies using their best threshold models
# Use the predictions already stored in all_results (which has probabilities)
strategy_probas = {}

# We need to re-train each strategy briefly to get probabilities
# Use the model factory approach from the notebook
from catboost import CatBoostClassifier as _CB
from xgboost import XGBClassifier as _XGB
from lightgbm import LGBMClassifier as _LGB

def _quick_model():
    """Return a fast model for comparison."""
    return _LGB(n_estimators=100, random_state=random_state, verbosity=-1, n_jobs=-1)

# Use stored results' probas if available, else recompute
print("Collecting PR curves for all strategies...")
strat_pr_curves = {}

for sname in strat_names:
    if sname not in all_results:
        continue
    sampler, use_cw = strategies[sname]
    spw = scale_pos_weight if use_cw else 1.0

    try:
        if sampler is not None:
            X_res, y_res = sampler.fit_resample(X_train, y_train)
        else:
            X_res, y_res = X_train.copy(), y_train.copy()

        m = _LGB(n_estimators=100, random_state=random_state, verbosity=-1, n_jobs=-1,
                 scale_pos_weight=spw)
        m.fit(X_res, y_res)
        y_prob = m.predict_proba(X_test)[:, 1]
        prec, rec, thresh = _prc2(y_test, y_prob)
        ap = float(_ap(y_test, y_prob))
        strat_pr_curves[sname] = {'precision': prec, 'recall': rec, 'ap': ap}
    except Exception as e:
        print(f"  Skipping {sname}: {e}")

# Plot full PR curves
fig, ax = plt.subplots(figsize=(10, 7))
cmap_strat = plt.cm.tab10
n_strats = len(strat_pr_curves)

for i, (sname, curve_data) in enumerate(strat_pr_curves.items()):
    color = cmap_strat(i / max(n_strats - 1, 1))
    ap = curve_data['ap']
    ax.plot(curve_data['recall'], curve_data['precision'],
            linewidth=2, color=color,
            label=f'{sname} (AP={ap:.3f})',
            alpha=0.85)

# Baseline: always predict positive (recall=1, precision=prevalence)
baseline_prec = float(y_test.mean())
ax.axhline(baseline_prec, color='grey', linestyle='--', linewidth=1.5,
           label=f'Random baseline (prec={baseline_prec:.3f})')

# Highlight operating points at recall=0.8 (high recall regime)
recall_target = 0.8
print(f"\nPrecision at recall \u2265 {recall_target} for each strategy:")
for sname, curve_data in strat_pr_curves.items():
    prec_arr = curve_data['precision']
    rec_arr  = curve_data['recall']
    # Find precision at closest recall to target
    idx = np.argmin(np.abs(rec_arr - recall_target))
    print(f"  {sname:<20}: precision={prec_arr[idx]:.3f} at recall={rec_arr[idx]:.3f}")

ax.axvline(recall_target, color='black', linestyle=':', alpha=0.4,
           label=f'Recall={recall_target} reference')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — All Sampling Strategies\n'
             '(Crossover points reveal which strategy is better in each recall regime)', fontsize=12)
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.05)

plt.tight_layout()
_pr_path = os.path.join(plots_dir, 'strategy_pr_curves.png')
fig.savefig(_pr_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {_pr_path}')
print('\nKey insight: the best strategy by PR-AUC may not be best in your operating regime.')
print('If you must maintain recall > 0.9, check which curve has highest precision there.')

## 6 — Metrics JSON Output

In [ ]:
# ---------------------------------------------------------------------------
# Write metrics JSON
# ---------------------------------------------------------------------------

RUN_END = datetime.now(timezone.utc)

all_strategies_metrics = {}
for strat_name, res in all_results.items():
    all_strategies_metrics[strat_name] = {
        'precision': res['precision'],
        'recall':    res['recall'],
        'f1':        res['f1'],
        'roc_auc':   res['roc_auc'],
        'pr_auc':    res['pr_auc'],
        'threshold': res['threshold'],
    }

output_metrics = {
    'run_metadata': {
        'run_start_utc':        RUN_START.isoformat(),
        'run_end_utc':          RUN_END.isoformat(),
        'duration_seconds':     (RUN_END - RUN_START).total_seconds(),
        'target_col':           target_col,
        'imbalance_ratio':      imbalance_ratio,
        'base_model':           base_model,
        'optuna_n_trials':      optuna_n_trials,
        'optimize_metric':      optimize_metric,
        'n_train':              len(X_train),
        'n_test':               len(X_test),
        'positive_rate_train':  float(y_train.mean()),
        'positive_rate_test':   float(y_test.mean()),
    },
    'all_strategies': all_strategies_metrics,
    'best_strategy': {
        'name':      best_strategy,
        'optuna_best_value': float(study.best_value),
        'optuna_best_params': study.best_params,
        'tuned_threshold': tuned_threshold,
        'metrics': {
            'precision': tuned_metrics['precision'],
            'recall':    tuned_metrics['recall'],
            'f1':        tuned_metrics['f1'],
            'roc_auc':   tuned_metrics['roc_auc'],
            'pr_auc':    tuned_metrics['pr_auc'],
        },
    },
}

Path(metrics_json_path).parent.mkdir(parents=True, exist_ok=True)
with open(metrics_json_path, 'w') as f:
    json.dump(output_metrics, f, indent=2)

print(f'Metrics saved to: {metrics_json_path}')
print(json.dumps(output_metrics, indent=2)[:1500])

## 7 — Summary

In [ ]:
# ---------------------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------------------

print('=' * 72)
print('IMBALANCED CLASSIFICATION — RUN SUMMARY')
print('=' * 72)
print(f'Base model    : {base_model}')
print(f'Optimize      : {optimize_metric}')
print(f'Optuna trials : {optuna_n_trials}')
print(f'Imbalance ratio (actual): {n_pos/n_total:.4f}  ({n_pos}/{n_total})')
print()
print('All strategies ranked by PR-AUC:')
print(f'{"Rank":<5} {"Strategy":<15} {"PR-AUC":>8} {"ROC-AUC":>8} {"F1":>8} {"Precision":>10} {"Recall":>8}')
print('-' * 72)

sorted_strategies = sorted(strat_names, key=lambda s: all_results[s]['pr_auc'], reverse=True)
for rank, name in enumerate(sorted_strategies, 1):
    res = all_results[name]
    marker = ' <<< BEST' if name == best_strategy else ''
    print(f'{rank:<5} {name:<15} {res["pr_auc"]:>8.4f} {res["roc_auc"]:>8.4f} '
          f'{res["f1"]:>8.4f} {res["precision"]:>10.4f} {res["recall"]:>8.4f}{marker}')

print()
print(f'Best strategy : {best_strategy!r}')
print(f'  After Optuna tuning ({optuna_n_trials} trials):')
print(f'  PR-AUC    : {tuned_metrics["pr_auc"]:.4f}')
print(f'  ROC-AUC   : {tuned_metrics["roc_auc"]:.4f}')
print(f'  F1        : {tuned_metrics["f1"]:.4f}')
print(f'  Precision : {tuned_metrics["precision"]:.4f}')
print(f'  Recall    : {tuned_metrics["recall"]:.4f}')
print(f'  Threshold : {tuned_threshold:.2f}')
print()
print(f'Model saved   : {model_output_path}')
print(f'Metrics saved : {metrics_json_path}')
print(f'Plots dir     : {plots_dir}')
print(f'Run duration  : {(RUN_END - RUN_START).total_seconds():.1f}s')
print('=' * 72)